In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

plt.style.use("ggplot")

import warnings
warnings.filterwarnings("ignore")

fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

nav = pd.read_csv("../data/raw/02_nav_history.csv")

aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")

category = pd.read_csv("../data/raw/05_category_inflows.csv")

folios = pd.read_csv("../data/raw/06_industry_folio_count.csv")

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

portfolio = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

# convert date columns to datetime format
nav['date']=pd.to_datetime(nav['date'])

aum['date']=pd.to_datetime(aum['date'])

sip['month']=pd.to_datetime(sip['month'])

category['month']=pd.to_datetime(category['month'])

folios['month']=pd.to_datetime(folios['month'])

transactions['transaction_date']=pd.to_datetime(
    transactions['transaction_date']
)

portfolio['portfolio_date']=pd.to_datetime(
    portfolio['portfolio_date']
)

benchmark['date']=pd.to_datetime(benchmark['date'])
# -----------------------------------------------------------------------------
# TASK 1 :  NAV Trend Analysis

# # Merge NAV with Fund Master
nav_df = nav.merge(
    fund_master[['amfi_code','scheme_name']],
    on='amfi_code',
    how='left'
)
# Plot
fig = px.line(
    nav_df,
    x='date',
    y='nav',
    color='scheme_name',
    title='Daily NAV Trend (2022–2026)'
)
# Highlight 2023 Bull Run
fig.add_vrect(
    x0="2023-01-01",
    x1="2023-12-31",
    fillcolor="green",
    opacity=0.15,
    annotation_text="Bull Run"
)
# Highlight 2024 Correction
fig.add_vrect(
    x0="2024-01-01",
    x1="2024-12-31",
    fillcolor="red",
    opacity=0.15,
    annotation_text="Correction"
)
fig.show()

fig.write_image("../reports/charts/nav_trend.png")

# Insight 1: Daily NAV Trend Analysis

The daily Net Asset Value (NAV) of most mutual fund schemes showed a consistent upward trend during the 2023 bull market, reflecting strong equity market performance. In contrast, 2024 exhibited increased volatility and slower NAV growth, indicating a period of market correction before stabilizing in 2025.

**Supporting Chart:** Daily NAV Trend (2022–2026)

In [ ]:
# task 2 : AUM Growth by Fund House AUM growth bar chart — grouped bar by fund house for each year 2022–2025. Highlight SBI at ₹12.5L Cr dominance using Seaborn.
# Create Year column
aum['Year']=aum['date'].dt.year

# Plot
plt.figure(figsize=(14,7))

sns.barplot(
    data=aum,
    x='Year',
    y='aum_lakh_crore',
    hue='fund_house'
)

plt.title("Fund House AUM Growth")

# Highlight SBI dominance
plt.axhline(
    y=12.5,
    color='red',
    linestyle='--'
)

plt.text(
    2,
    12.7,
    "SBI ₹12.5 Lakh Cr"
)
plt.savefig("../reports/charts/aum_growth.png")


# Insight 2: Assets Under Management (AUM) Growth

Assets Under Management increased across major fund houses during the analysis period. SBI Mutual Fund maintained the industry's highest AUM, surpassing ₹12.5 lakh crore by 2025, highlighting its strong market presence and investor confidence.

**Supporting Chart:** Fund House AUM Growth (2022–2025)

In [ ]:
# task3: SIP Inflow Trend SIP inflow time-series — monthly SIP trend Jan 2022 – Dec 2025. Annotate the ₹31,002 Cr all-time high (Dec 2025) using Plotly.
# Plot
fig = px.line(
    sip,
    x='month',
    y='sip_inflow_crore',
    markers=True
)
highest=sip.loc[
    sip['sip_inflow_crore'].idxmax()
]

fig.add_annotation(
    x=highest['month'],
    y=highest['sip_inflow_crore'],
    text="₹31,002 Cr",
    showarrow=True
)
fig.show()

print(highest)
print(type(highest['month']))

highest = sip.loc[sip['sip_inflow_crore'].idxmax()]

fig.add_annotation(
    x=highest['month'].to_pydatetime(),
    y=highest['sip_inflow_crore'],
    text="₹31,002 Cr",
    showarrow=True,
    arrowhead=2
)
fig.add_annotation(
    x=str(highest['month']),
    y=highest['sip_inflow_crore'],
    text="₹31,002 Cr",
    showarrow=True
)
sip['month'] = pd.to_datetime(sip['month'])

fig = px.line(
    sip,
    x='month',
    y='sip_inflow_crore',
    markers=True
)

highest = sip.loc[sip['sip_inflow_crore'].idxmax()]

fig.add_annotation(
    x=highest['month'].strftime('%Y-%m-%d'),
    y=highest['sip_inflow_crore'],
    text=f"₹{highest['sip_inflow_crore']:,.0f} Cr",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40
)

fig.show()
fig.write_image("../reports/charts/sip_trend.png")


# Insight 3: Monthly SIP Inflow Trend

Monthly Systematic Investment Plan (SIP) inflows displayed a steady upward trend throughout the study period. SIP investments reached an all-time high of ₹31,002 crore in December 2025, reflecting increasing retail participation and long-term investment discipline.

**Supporting Chart:** Monthly SIP Inflow Trend

In [ ]:
# task 4: Category inflow heatmap — months on X-axis, fund categories on Y-axis, net inflow as colour intensity using Seaborn.
# Category Inflow Heatmap
heat=category.pivot_table(
    index='category',
    columns=category['month'].dt.strftime('%b-%Y'),
    values='net_inflow_crore'
)
plt.figure(figsize=(16,8))

sns.heatmap(
    heat,
    cmap='RdYlGn',
    annot=True
)

plt.title("Category Inflows")
plt.savefig("../reports/charts/category_heatmap.png")

# Insight 4: Category-wise Investment Preferences

Investment inflows varied significantly across mutual fund categories over time. Equity-oriented categories consistently attracted higher net inflows than debt-oriented categories, indicating investors' preference for long-term wealth creation through equity markets.

**Supporting Chart:** Category Inflow Heatmap

In [ ]:
# TASK 5: Investor Demographics — age group distribution pie chart. SIP amount box plot by age group. Gender split.
# pie chart for age group distribution
transactions['age_group'].value_counts().plot.pie(
    autopct='%1.1f%%'
)
# sip amount box plot by age group
sns.boxplot(
    data=transactions,
    x='age_group',
    y='amount_inr'
)
# gender split bar chart
sns.countplot(
    data=transactions,
    x='gender'
)
plt.savefig("../reports/charts/investor_demographics.png")



# Insight 5: Investor Demographics by Age

The majority of investors belong to the working-age population, indicating that salaried individuals form the largest contributor to mutual fund investments. This demographic also demonstrates higher participation in systematic investment plans.

**Supporting Chart:** Age Group Distribution Pie Chart

In [ ]:
# – Gender Distribution
plt.figure(figsize=(6,5))

sns.countplot(
    data=transactions,
    x='gender'
)

plt.title("Gender-wise Investor Count")

plt.xlabel("Gender")

plt.ylabel("Number of Investors")

plt.savefig("../reports/charts/gender_bar.png",
            dpi=300,
            bbox_inches='tight')

plt.show()

## Insight 6: Gender-wise Participation

The analysis indicates that mutual fund participation is higher among male investors, while female investors also represent a significant share of the investor base.

**Supporting Chart:** Gender-wise Investor Distribution

In [ ]:
# Task 7: Geographic distribution — horizontal bar chart of SIP amount by state. T30 vs B30 city tier pie chart.
# state-wise SIP amount horizontal bar chart
state = (
    transactions.groupby('state')['amount_inr']
    .sum()
    .sort_values()
)

state.plot.barh(figsize=(12,8))
# city tier pie chart
transactions['city_tier'].value_counts().plot.pie(
    autopct='%1.1f%%'
)
plt.savefig("../reports/charts/geographic_distribution.png")

# Insight 7: Geographic Contribution to SIP Investments

Investment activity is concentrated in financially developed states, contributing the largest share of SIP investments. The comparison between T30 and B30 cities also indicates growing participation from smaller cities, reflecting increasing financial awareness across India.

**Supporting Charts:** State-wise SIP Investment Bar Chart and T30 vs B30 Pie Chart

In [ ]:
# task 8: Folio count growth — line chart from 13.26 Cr (Jan 2022) to 26.12 Cr (Dec 2025). Mark key milestones.
# Folio Growth
fig=px.line(
    folios,
    x='month',
    y='total_folios_crore',
    markers=True
)
fig.add_annotation(
    x="2022-01-01",
    y=13.26,
    text="13.26 Cr"
)

fig.add_annotation(
    x="2025-12-01",
    y=26.12,
    text="26.12 Cr"
)
fig.show()
plt.savefig("../reports/charts/folio_growth.png")

# Insight 8: Growth in Mutual Fund Folios

The total number of mutual fund folios nearly doubled from 13.26 crore in January 2022 to 26.12 crore in December 2025. This substantial increase highlights rapid expansion in retail investor participation and sustained growth in the mutual fund industry.

**Supporting Chart:** Industry Folio Growth Trend

In [ ]:

# Task 9: NAV return correlation matrix — compute pairwise correlation of daily returns for 10 selected funds. Seaborn heatmap.
nav_return = nav.merge(
    fund_master[['amfi_code','scheme_name']],
    on='amfi_code'
)

pivot = nav_return.pivot(
    index='date',
    columns='scheme_name',
    values='nav'
)

returns = pivot.pct_change()
corr = returns.iloc[:,0:10].corr()
plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)
plt.title("NAV Return Correlation Matrix")
plt.savefig("../reports/charts/nav_correlation.png")

# Insight 9: NAV Return Correlation Analysis

The correlation matrix compares the daily returns of the selected mutual fund schemes to identify how closely they move together. Most funds belonging to similar investment categories exhibit strong positive correlations, indicating similar market behaviour, while lower correlations suggest diversification opportunities that can help reduce portfolio risk.

**Supporting Chart:** NAV Return Correlation Matrix (Heatmap)

In [ ]:
# Task 10 : Sector allocation donut — aggregate sector weights from portfolio_holdings.csv across all equity funds.
sector = portfolio.groupby(
    'sector'
)['weight_pct'].sum()
fig=px.pie(
    values=sector.values,
    names=sector.index,
    hole=0.55
)
plt.title("Sector Allocation Donut")
plt.savefig("../reports/charts/sector_allocation_donut.png")
plt.show()

# Insight 10: Sector Allocation Across Equity Mutual Funds

The sector allocation analysis provides an overview of how equity mutual fund assets are distributed across various industries. The portfolio demonstrates a diversified investment strategy, with higher allocations toward sectors that have historically contributed to long-term economic growth. Such diversification helps improve portfolio stability while reducing the impact of sector-specific market fluctuations.

**Supporting Chart:** Sector Allocation Donut Chart